# Radon vector-deskew lab

Reads a **manual label file**, pulls each label's **exact vectors** (via
`LabelEntry.vector_signatures` -> `label_schema.path_signature`), and sweeps a Radon-style
angle `theta` **purely on vector geometry** -- no rasterization anywhere.

### The projection (one fixed function, `project(vectors, theta)`)

1. rotate every vector's points by `-theta` about the cluster centre
2. take the cluster's rotated-bbox y-range `[Y0, Y1]`
3. cut it into **101 equal intervals** -> the **100 internal division lines**
   `y_k = Y0 + k*(Y1-Y0)/101`, `k = 1..100` (first / last edge excluded)
4. for each **horizontal** line `y = y_k`, count how many vectors it **geometrically
   intersects** (`l` -> segment, `c` -> Bernstein-flattened sub-segments, `re`/`qu` -> the 4
   rotated edges); each vector counts 0 or 1
5. -> a length-100 array = the 1-D projection / histogram at that `theta`

### Value functions (`VALUES`, pluggable)

Each takes the length-100 array and returns one scalar. Add one by appending to `VALUES`
(and `OPT` with `"max"` / `"min"` for the argopt marker), then re-run from the sweep cell.

## 0 - Config

In [ ]:
from pathlib import Path

LABEL_PATH        = None     # None -> first (name-sorted) outputs/labels/*.json
PDF_PATH_OVERRIDE = None     # None -> LabelSet.pdf_path (resolved vs repo root)
PAGE_INDEX        = None     # None -> every page present in the label file

THETA_MIN, THETA_MAX, THETA_STEP = -90.0, 90.0, 2.0
N_LINES        = 100         # division lines per projection (task spec: 100)
CURVE_SAMPLES  = 24          # sub-segments a cubic 'c' item is flattened into
GRID_THETA_COLS = 9          # theta columns in the per-cluster grid

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Labels -> the exact labelled vector set per label

`extract_vectors(page)` once per page (raw `get_drawings()` geometry -- no classification,
no clustering, no rendering), keyed by `path_signature`; each label pulls its own vectors by
the signatures it stored.

In [ ]:
import numpy as np
from rastervec.Reader.reader import Reader
from rastervec.pipelines._steps import extract_vectors
from rastervec.Evaluation.Labelling.label_schema import (
    load_labels, split_labelset_by_source, path_signature)
from rastervec.helpers.geometry import union_bbox
from rastervec.paths import output_dir, REPO_ROOT


def _resolve_pdf(raw):
    p = Path(str(raw).replace(chr(92), "/"))
    if p.is_file():
        return p
    for base in (Path.cwd(), REPO_ROOT):
        for cand in ((base / p), (base / p.name)):
            if cand.is_file():
                return cand.resolve()
    for sub in ("references", "references2"):
        cand = REPO_ROOT / sub / p.name
        if cand.is_file():
            return cand
    raise FileNotFoundError(f"cannot locate PDF {raw!r} (cwd={Path.cwd()}, repo={REPO_ROOT})")


def _pick_label_path():
    if LABEL_PATH:
        return Path(LABEL_PATH)
    cands = sorted(Path(output_dir("labels")).glob("*.json"))
    if not cands:
        raise FileNotFoundError("no outputs/labels/*.json; set LABEL_PATH")
    return cands[0]


label_path = _pick_label_path()
labels = load_labels(label_path)
pdf_path = _resolve_pdf(PDF_PATH_OVERRIDE or labels.pdf_path)
manual = split_labelset_by_source(labels)["manual"].entries
print(f"label file: {label_path}")
print(f"pdf:        {pdf_path}")
print(f"manual entries: {len(manual)}")

by_page = {}
for e in manual:
    if PAGE_INDEX is None or e.page_index == PAGE_INDEX:
        by_page.setdefault(e.page_index, []).append(e)

matched = []          # (entry, [Vector, ...])
with Reader(pdf_path) as r:
    for pidx in sorted(by_page):
        page_vectors = extract_vectors(r.get_page(pidx))
        sigmap = {path_signature(v): v for v in page_vectors}
        for e in by_page[pidx]:
            sel = [sigmap[s] for s in e.vector_signatures if s in sigmap]
            matched.append((e, sel))
            print(f"  p{pidx}  {e.text!r:12}  rot={e.expected_rotation:>4}  "
                  f"found {len(sel)}/{len(e.vector_signatures)} vectors")

print(f"\nmatched clusters: {len(matched)}")

## 3 - Geometry helpers

In [ ]:
from rastervec.helpers.geometry import item_points


def rotate_pts(pts, theta_deg, centre):
    """Rotate (N,2) points by -theta about centre (radon-ray frame)."""
    t = np.deg2rad(-theta_deg)
    c, s = np.cos(t), np.sin(t)
    m = np.array([[c, -s], [s, c]])
    return (np.asarray(pts, float) - centre) @ m.T + centre


def _bezier(pts, n):
    p0, p1, p2, p3 = [np.asarray(p, float) for p in pts]
    u = np.linspace(0.0, 1.0, n + 1)[:, None]
    b = ((1 - u) ** 3 * p0 + 3 * (1 - u) ** 2 * u * p1
         + 3 * (1 - u) * u ** 2 * p2 + u ** 3 * p3)
    return list(zip(b[:-1], b[1:]))


def item_subsegments(item, curve_samples=None):
    """Straight (a, b) pieces approximating one Vector.items entry:
    'l' -> 1 segment, 'c' -> `curve_samples` Bernstein chords,
    're'/'qu' -> the 4 closed border edges."""
    n = CURVE_SAMPLES if curve_samples is None else curve_samples
    k = item[0]
    if k == "l":
        return [(item[1], item[2])]
    if k == "c":
        return _bezier(item_points(item), n)
    if k == "qu":
        c = [tuple(p) for p in item_points(item)]
    elif k == "re":
        x0, y0, x1, y1 = tuple(item[1])
        c = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]
    else:
        return []
    return list(zip(c, c[1:] + c[:1]))


def cluster_centre(vectors):
    x0, y0, x1, y1 = union_bbox([v.bbox for v in vectors])
    return np.array([(x0 + x1) / 2.0, (y0 + y1) / 2.0])


def rotated_segments(vectors, theta_deg):
    """Rotate the whole cluster by -theta about its centre.
    Returns (per_vector, Y0, Y1) where per_vector[i] is an (M_i, 2, 2)
    array of that vector's sub-segment endpoints in the rotated frame, and
    [Y0, Y1] is the cluster's rotated y-range."""
    centre = cluster_centre(vectors)
    per_vector = []
    for v in vectors:
        segs = [ab for it in v.items for ab in item_subsegments(it)]
        if not segs:
            per_vector.append(np.empty((0, 2, 2)))
            continue
        pts = rotate_pts(np.array(segs, float).reshape(-1, 2), theta_deg, centre)
        per_vector.append(pts.reshape(-1, 2, 2))
    ys = np.concatenate([pv[:, :, 1].ravel() for pv in per_vector if pv.size]) \
        if any(pv.size for pv in per_vector) else np.array([0.0])
    return per_vector, float(ys.min()), float(ys.max())

## 4 - The projection function

In [ ]:
def project(vectors, theta_deg):
    """(line_y[N_LINES], profile[N_LINES]) -- 100 horizontal rays y=y_k across
    the rotated cluster; profile[k] = how many vectors ray k intersects."""
    per_vector, Y0, Y1 = rotated_segments(vectors, theta_deg)
    k = np.arange(1, N_LINES + 1)
    line_y = Y0 + k * (Y1 - Y0) / (N_LINES + 1)
    prof = np.zeros(N_LINES)
    if Y1 - Y0 < 1e-9:
        return line_y, prof
    for pv in per_vector:
        if not pv.size:
            continue
        ylo = pv[:, :, 1].min(axis=1)[:, None]
        yhi = pv[:, :, 1].max(axis=1)[:, None]
        hit = ((ylo <= line_y[None, :]) & (line_y[None, :] <= yhi)).any(axis=0)
        prof += hit
    return line_y, prof

## 5 - Value functions  `fn(profile[100]) -> float`

In [ ]:
from rastervec.OCR import radon as R


def val_postl_sq(a):
    return float(np.sum(np.asarray(a, float) ** 2))


def val_gap_objective(a):
    return R._skew_objective(np.asarray(a, float))     # pure profile math, no raster


def val_variance(a):
    return float(np.var(np.asarray(a, float)))


VALUES = {
    "postl_sq":      val_postl_sq,
    "gap_objective": val_gap_objective,
    "variance":      val_variance,
}
OPT = {"postl_sq": "max", "gap_objective": "min", "variance": "max"}

## 6 - Sweep **every** cluster in the label file

`sweep[ci]` holds that cluster's `entry`, `vectors`, per-theta `profiles`, and per-value
`curves` over the shared `thetas` grid.

In [ ]:
thetas = np.arange(THETA_MIN, THETA_MAX + THETA_STEP / 2, THETA_STEP)

sweep = []
for ci, (e, vec) in enumerate(matched):
    profiles = {float(th): project(vec, float(th))[1] for th in thetas} if vec else {}
    curves = {n: np.array([fn(profiles[float(th)]) for th in thetas]) for n, fn in VALUES.items()} \
        if vec else {}
    sweep.append(dict(entry=e, vectors=vec, profiles=profiles, curves=curves))
    print(f"#{ci:<3} {e.text!r:14} rot={e.expected_rotation:>4}  {len(vec)} vectors"
          + ("" if vec else "   <-- no vectors, skipped"))

## 7 - Per cluster: rotated-cluster render + 1-D projection at every theta

Two rows per cluster -- top: the cluster's vectors rotated by `-theta` (pure line render,
the 100 division lines dotted); bottom: the projection histogram at that theta.

In [ ]:
import matplotlib.pyplot as plt

sel = thetas[np.linspace(0, len(thetas) - 1, min(GRID_THETA_COLS, len(thetas))).astype(int)]
ncol = len(sel)

for ci, s in enumerate(sweep):
    vec, e = s["vectors"], s["entry"]
    if not vec:
        continue
    fig, axes = plt.subplots(2, ncol, figsize=(1.8 * ncol, 4.2), squeeze=False,
                             gridspec_kw=dict(height_ratios=[2, 1]))
    fig.suptitle(f"#{ci}  {e.text!r}   expected_rotation={e.expected_rotation}"
                 f"   ({len(vec)} vectors)", fontsize=10)
    for j, th in enumerate(sel):
        ti = int(np.argmin(np.abs(thetas - th)))
        prof = s["profiles"][float(th)]
        per_vector, Y0, Y1 = rotated_segments(vec, float(th))
        line_y = Y0 + np.arange(1, N_LINES + 1) * (Y1 - Y0) / (N_LINES + 1)

        top = axes[0][j]
        for pv in per_vector:
            for (a, b) in pv:
                top.plot([a[0], b[0]], [a[1], b[1]], color="k", lw=0.5)
        for ly in line_y:
            top.axhline(ly, color="tab:blue", lw=0.3, alpha=0.35)
        top.set_aspect("equal"); top.invert_yaxis()
        top.set_xticks([]); top.set_yticks([])
        top.set_title(f"theta={th:+.0f}", fontsize=8)

        bot = axes[1][j]
        bot.bar(np.arange(N_LINES), prof, width=1.0)
        bot.set_xticks([]); bot.set_yticks([])
        bot.set_xlabel("\n".join(f"{n}={s['curves'][n][ti]:.3g}" for n in VALUES), fontsize=6)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

### 7b - (optional) interactive slider  — needs `ipywidgets`

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display

    def _show(ci, ti):
        s = sweep[ci]
        if not s["vectors"]:
            print("no vectors for this cluster"); return
        th = float(thetas[ti])
        per_vector, Y0, Y1 = rotated_segments(s["vectors"], th)
        line_y = Y0 + np.arange(1, N_LINES + 1) * (Y1 - Y0) / (N_LINES + 1)
        fig, (top, bot) = plt.subplots(2, 1, figsize=(11, 6),
                                       gridspec_kw=dict(height_ratios=[2, 1]))
        for pv in per_vector:
            for (a, b) in pv:
                top.plot([a[0], b[0]], [a[1], b[1]], color="k", lw=0.6)
        for ly in line_y:
            top.axhline(ly, color="tab:blue", lw=0.3, alpha=0.35)
        top.set_aspect("equal"); top.invert_yaxis()
        top.set_title(f"#{ci} {s['entry'].text!r}   theta={th:+.1f}   "
                      + "   ".join(f"{n}={s['curves'][n][ti]:.4g}" for n in VALUES), fontsize=10)
        bot.bar(np.arange(N_LINES), s["profiles"][th], width=1.0)
        plt.tight_layout(); plt.show()

    display(W.interactive(
        _show,
        ci=W.IntSlider(min=0, max=len(sweep) - 1, value=0, description="cluster"),
        ti=W.IntSlider(min=0, max=len(thetas) - 1, value=len(thetas) // 2, description="theta idx")))
except ImportError:
    print("ipywidgets not installed - use the grid above")

## 8 - Value vs theta, per cluster

In [ ]:
for ci, s in enumerate(sweep):
    if not s["vectors"]:
        continue
    e = s["entry"]
    fig, axes = plt.subplots(len(VALUES), 1, figsize=(11, 2.4 * len(VALUES)), squeeze=False)
    fig.suptitle(f"#{ci}  {e.text!r}   expected_rotation={e.expected_rotation}", fontsize=10)
    for ax, name in zip(axes[:, 0], VALUES):
        arr = s["curves"][name]
        fin = np.isfinite(arr)
        ax.plot(thetas[fin], arr[fin], marker=".", lw=1)
        if fin.any():
            opt = OPT.get(name, "max")
            ti = (np.nanargmax(np.where(fin, arr, -np.inf)) if opt == "max"
                  else np.nanargmin(np.where(fin, arr, np.inf)))
            ax.axvline(thetas[ti], color="tab:green", ls="--",
                       label=f"arg{opt} theta={thetas[ti]:+.1f}")
        if e.expected_rotation is not None:
            ax.axvline(e.expected_rotation, color="tab:red", ls=":", label="expected_rotation")
        ax.set_title(f"{name}   ({OPT.get(name, 'max')})", fontsize=9)
        ax.set_xlabel("theta (deg)"); ax.legend(fontsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()